# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment (Python 3.8 or newer), on Windows, macOS and Linux. It needs no Docker: everything here is plain Python.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy scipy soundfile matplotlib imageio-ffmpeg --quiet

**Imports**

In [ ]:
# --- Imports: libraries and functions used in this notebook ---
import os
import numpy as np
from IPython.display import Audio, display
from Audio import fsk_modem as fm
from Audio.audio_watermark import simulate_channel, capacity

### 1. Building an Acoustic Modem

The watermark in `Audio_Transmission.ipynb` hides a message *inside* music: inaudible, but slow. A **modem** does the opposite: it turns data into sound you can clearly hear, and in exchange it is much faster. Dial-up internet, fax machines and the GibberLink demonstration all work this way.

This modem uses two ideas from the lecture:

- **4-FSK** (frequency-shift keying): each symbol is one of four tones, so one symbol carries two bits: `00` → tone 0 (lowest), `01` → tone 1, `10` → tone 2, `11` → tone 3.
- **A chirp frame**: the receiver has no idea when you pressed *play*. Two known sweeps mark the start and the end, and the receiver finds them by correlation, even below the noise.

```text
[ silence ][ up-chirp 0.5 s ][ gap ][ payload: 4-FSK symbols ][ gap ][ down-chirp 0.25 s ][ silence ]
```

The transmitter, the chirp search and the slicing into symbols are given. **The detector, which decides which tone was sent in each symbol, is yours to write.**

#### 1.1 Your Channel

Every student transmits on their own tones and symbol length, derived from the last two digits of the matriculation number:

- base frequency = 800 Hz + 100 Hz × (last digit)
- symbol length = 30 ms + 5 ms × (second-last digit mod 7)
- tone spacing = 200 Hz

In [ ]:
# INPUT
LEGI = "00-000-000"          # <-- Change this. Your matriculation number

params = fm.my_parameters(LEGI)
print(params)

#### 1.2 The Transmitter

The cell below writes your matriculation number into a frame and saves it as `Modem_TX.wav`. Listen to it, and look at its spectrogram: the rising and falling lines are the chirps, the short horizontal dashes in between are the four tones.

In [ ]:
MESSAGE = LEGI                # the text to transmit
TX_FILE = "Modem_TX.wav"

tx = fm.make_frame(MESSAGE, params)
fm.save_wav(TX_FILE, tx)
print(f"{TX_FILE}: {len(tx) / fm.SAMPLE_RATE:.2f} s, {8 * len(MESSAGE)} bits")
display(Audio(tx, rate=fm.SAMPLE_RATE))
fm.plot_spectrogram(tx, fm.SAMPLE_RATE, params, title="Transmitted frame (dashed lines: your four tones)")

#### 1.3 The Detector (your task)

After the chirps are found, the payload is cut into one short array per symbol (the edges of each symbol are dropped, because that is where one tone changes into the next). For each symbol the receiver must decide which of the four tones is present.

**Step 1: `tone_energies(segment, rate, tones)`.** For one symbol `segment` (samples $x[n]$, $n = 0 \dots N-1$), return one number per tone $f_k$ that measures how much of that tone is present. The magnitude of a single DFT coefficient does exactly this:

$$
E_k = \left|\sum_{n=0}^{N-1} x[n]\, e^{-j 2\pi f_k n / f_s}\right|
$$

Taking the magnitude makes this a **non-coherent** detector: it does not care about the phase of the tone, which over the air is unknown anyway.

**Step 2: `demodulate(energies)`.** `energies` has one row per symbol and one column per tone. For each row, pick the strongest tone (index 0 to 3) and turn it back into two bits, most significant first. Return one flat array of bits.

Replace the `raise NotImplementedError` lines with your code, then run the self-test. It decodes three known frames: a clean one, a noisy one at 44.1 kHz, and one with different tones and symbol length.

In [ ]:
def tone_energies(segment, rate, tones):
    """Return an array with one energy value per tone in `tones` (Hz) for this symbol."""
    # YOUR CODE HERE
    raise NotImplementedError("Write tone_energies (Section 1.3)")


def demodulate(energies):
    """energies: array (symbols x 4). Return the bits as one flat array, two bits per symbol."""
    # YOUR CODE HERE
    raise NotImplementedError("Write demodulate (Section 1.3)")

In [ ]:
fm.self_test(tone_energies, demodulate)

#### 1.4 Over the Air

Play `Modem_TX.wav` on your laptop and record it with your phone at about 30 cm, the same way you recorded the mystery clip. Save the recording next to this notebook and set its name below. `.m4a`, `.mp3` and `.wav` all work, at any sample rate.

Start the recording **before** you press play and stop it after the sound ends. The chirps tell the receiver where the message is, so the extra silence costs nothing.

If no recording is found, the cell decodes a **simulated** 50 cm recording instead, so you can test your detector. Do not submit simulated results.

In [ ]:
# INPUT
RECORDING_FILE = "Modem_Recorded.m4a"     # <-- Change this. Your phone recording

if os.path.exists(RECORDING_FILE):
    source, label = RECORDING_FILE, "your recording"
else:
    source = simulate_channel(TX_FILE, "SIMULATED_Modem_50cm.wav", "50cm")
    label = "SIMULATED 50 cm recording - do not submit"
    print(f"{RECORDING_FILE} not found: using a simulated recording.\n")

rx, rx_rate = fm.load_audio(source)
frame = fm.find_frame(rx, rx_rate)
fm.plot_spectrogram(rx, rx_rate, params, frame, title=f"Received: {label} (red: chirps found)")
received = fm.decode_frame(rx, rx_rate, params, tone_energies, demodulate)

correct = sum(a == b for a, b in zip(MESSAGE, received))
print(f"sent '{MESSAGE}', received '{received}': {correct}/{len(MESSAGE)} characters correct")

#### 1.5 Pushing the Link

Your personal symbol length is a comfortable starting point. Shorten it step by step: each halving doubles the bit rate. Regenerate the frame, play it, record it, and decode it again until the message breaks. Keep the distance the same while you do this, so that only one thing changes.

In [ ]:
# INPUT
SYMBOL_MS = params.symbol_ms / 2       # <-- Change this. Try shorter and shorter symbols

fast = fm.ModemParameters(params.base_hz, params.spacing_hz, SYMBOL_MS)
FAST_FILE = f"Modem_TX_{SYMBOL_MS:g}ms.wav"
fm.save_wav(FAST_FILE, fm.make_frame(MESSAGE, fast))
print(f"{FAST_FILE}: {fast}")

FAST_RECORDING = f"Modem_Recorded_{SYMBOL_MS:g}ms.m4a"   # <-- Change this. Its recording
if os.path.exists(FAST_RECORDING):
    x, r = fm.load_audio(FAST_RECORDING)
    fm.decode_frame(x, r, fast, tone_energies, demodulate)
else:
    print(f"Record {FAST_FILE} and save it as {FAST_RECORDING} to decode it.")

#### 1.6 Capacity vs. Reality

Shannon's capacity is the highest bit rate *any* scheme could reach over a channel of bandwidth $B$ and signal-to-noise ratio SNR:

$$
C = B \log_2(1 + \mathrm{SNR}), \qquad \mathrm{SNR} = 10^{\mathrm{SNR_{dB}}/10}
$$

Use the SNR you measured in `Audio_Analysis.ipynb` (Section 2) and, as $B$, the band your modem occupies: from half a spacing below the lowest tone to half a spacing above the highest, which is 4 × 200 Hz = 800 Hz. The cell compares $C$ with the bit rates you actually achieved.

In [ ]:
# INPUT
SNR_DB         = 20.0                   # <-- Change this. Your measured SNR (Audio_Analysis.ipynb, Section 2)
FASTEST_OK_MS  = params.symbol_ms       # <-- Change this. Shortest symbol length that still decoded (1.5)
WATERMARK_LEVEL = 2                     # the level you used in Audio_Transmission.ipynb

B = fm.TONE_COUNT * fm.SPACING_HZ
C = B * np.log2(1 + 10 ** (SNR_DB / 10))
fsk_rate = 2 * 1000 / FASTEST_OK_MS
wm_rate = capacity(WATERMARK_LEVEL, 61.5)["bits_per_second"]

print(f"bandwidth B          : {B:.0f} Hz")
print(f"Shannon capacity C   : {C:8.1f} bit/s  at {SNR_DB:.1f} dB SNR")
print(f"your FSK modem       : {fsk_rate:8.1f} bit/s  ({100 * fsk_rate / C:.1f} % of C)")
print(f"watermark, level {WATERMARK_LEVEL}   : {wm_rate:8.1f} bit/s  ({100 * wm_rate / C:.2f} % of C, but in the whole audio band and inaudible)")